# Baseline для аудита детектора

Метка ошибки построена сопоставлением с ручным боксом того же класса при IoU ≥ 0,60. Чем ниже confidence, тем выше приоритет проверки. Этот baseline не требует обучения. Ниже считаем AP отдельно по каждому размеченному дню и создаём файл отправки на Kaggle.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from IPython.display import display

kaggle_input = Path('/kaggle/input')
if os.environ.get('DATA_ROOT'):
    data = Path(os.environ['DATA_ROOT'])
elif kaggle_input.exists():
    datasets = [p.parent for p in sorted(kaggle_input.rglob('sample_submission.csv'))
                if (p.parent/'train.csv').is_file() and (p.parent/'test.csv').is_file()]
    assert len(datasets) == 1, 'Укажите DATA_ROOT с распакованным основным пакетом'
    data = datasets[0]
else:
    data = Path.cwd()
    if not (data/'sample_submission.csv').exists():
        data = data/'labeled_case2'/'kaggle'/'participant_data'
train = pd.read_csv(data/'train.csv', dtype={'id': str})
test = pd.read_csv(data/'test.csv', dtype={'id': str})
template = pd.read_csv(data/'sample_submission.csv', dtype={'id': str})
assert 'error' in train and 'error' not in test and 'Usage' not in test
assert train.id.is_unique and test.id.is_unique and set(train.id).isdisjoint(test.id)


## Проверка по размеченным дням

При обучении своей модели используйте один день для обучения и второй для валидации, затем поменяйте их местами. Сортировка по confidence считается отдельно на каждом дне. Постоянный балл даёт AP, равную доле ошибок. Столы исключены из оценки, как в тесте.

In [ ]:
frames = pd.read_csv(data/'train_frames.csv')
days = frames[['frame_id','timestamp']].copy()
days['day'] = pd.to_datetime(days.timestamp,utc=True,format='mixed').dt.date
validation = train[train.class_name != 'table'].merge(days[['frame_id','day']],on='frame_id',validate='many_to_one')
results = []
for day, part in validation.groupby('day'):
    results.append({'day':day,'frames':part.frame_id.nunique(),'boxes':len(part),
                    'errors':int(part.error.sum()),'constant_score_ap':part.error.mean(),
                    'confidence_ap':average_precision_score(part.error, 1-part.confidence)})
display(pd.DataFrame(results).round(4))
print('Средний AP по двум дням:',round(np.mean([row['confidence_ap'] for row in results]),4))


## Отправка

Одна строка для каждого id из шаблона. Kaggle отдельно посчитает AP на Public и Private.

In [ ]:
ranked = test[['id']].assign(score=1-test.confidence)
submission = template[['id']].merge(ranked,on='id',how='left',validate='one_to_one')
assert len(submission)==len(test)
assert np.isfinite(submission.score).all() and submission.score.between(0,1).all()
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
submission.to_csv(output_dir/'submission.csv',index=False)
print('submission.csv:',len(submission),'строк')
display(submission.head())
